# 2 — The solver, checked against an exact solution

`groundtruth` solves the scalar transmission problem
$(\nabla^2+k^2)\psi=0$ with $[\psi]=0$, $[\partial_n\psi]=0$ on a body of
revolution, using a Müller second-kind boundary-integral formulation reduced
to the $m=0$ azimuthal harmonic.

The homogeneous ball has a closed-form solution by separation of variables
(the scalar analogue of Mie), so it fixes the solver's error with no
reference to any optical model.  Two things are established here: the
convergence rate, and the range of electrical size where the solver is
usable.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import numpy as np, matplotlib.pyplot as plt
from nbstyle import *

import time
from groundtruth.bem import (sphere_generator, solve_muller, field_map,
                             field_interior)
from groundtruth.exact import ScalarBall

LAM = 1.0

## Convergence on the surface

Plane wave on a ball of radius $2\lambda$, $n_2=1.5$.  The error is the
maximum relative difference between the BEM surface field $u$ and the exact
series.

In [ ]:
def ball_error(a, n2, N, n1=1.0):
    b = ScalarBall(n1, n2, a, LAM)
    g = sphere_generator(a, N)
    if (g.n_rho * g.rho + g.n_z * g.z).mean() < 0:
        g.flip_normal()
    ui = np.exp(1j * b.k1 * g.z)
    vi = 1j * b.k1 * g.n_z * ui
    u, v, _ = solve_muller(g, b.k1, b.k2, ui, vi)
    ue = b.surface_field(g.z / a)
    return np.abs(u - ue).max() / np.abs(ue).max()

Ns = np.array([100, 200, 400, 800])
errs = np.array([ball_error(2.0, 1.5, int(N)) for N in Ns])
for N, e in zip(Ns, errs):
    print(f"N = {N:4d}   error = {e:.3e}")
p = np.polyfit(np.log(Ns[:3]), np.log(errs[:3]), 1)[0]
print(f"\nobserved order over the first three meshes: h^{-p:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 4.3))
ax.loglog(Ns, errs, "o-", color=BLUE, lw=2.2, ms=7, label="measured")
ax.loglog(Ns, errs[0] * (Ns / Ns[0]) ** -2.0, ls=(0, (6, 4)), color=MUTED,
          lw=1.6, label="$O(h^2)$")
ax.set_xlabel("panels N"); ax.set_ylabel("max rel. error in u")
ax.legend(fontsize=9)
ttl(ax, "Surface-field convergence", "ball a = 2λ, n₂ = 1.5, plane wave")
plt.show()

## The field inside, and the limit in electrical size

`method_check.npz` holds a run at $a=1.5\lambda$, $n_2=1.5$ (so
$k_2a=14.1$) plus a sweep of the surface error against $k_2a$ at two
contrasts.

In [ ]:
MC = load("method_check")
BL = load("ball_fields")

fig, ax = plt.subplots(1, 3, figsize=(15.5, 4.6))

im = meridional(ax[0], BL["rg"], BL["zg"], BL["psi"], 1.0, decades=1.8)
t = np.linspace(0, 2 * np.pi, 400)
ax[0].plot(float(BL["a"]) * np.cos(t), float(BL["a"]) * np.sin(t),
           color=EDGE, lw=1.6)
plt.colorbar(im, ax=ax[0], fraction=0.046, pad=0.03).set_label("log₁₀ |ψ|",
                                                               fontsize=8.5)
ttl(ax[0], "Ball, exact series",
    f"a = {float(BL['a']):.0f}λ, n₂ = {float(BL['n2'])}, "
    f"k₁a = {float(BL['x1']):.1f}, plane wave from −z")

ax[1].plot(MC["z_ax"], np.abs(MC["exact"]), color=GREEN, lw=3.0,
           label="exact series")
ax[1].plot(MC["z_ax"], np.abs(MC["bem"]), color=RED, lw=1.6, ls=(0, (5, 3)),
           label="Müller BOR-BEM")
ax[1].set_xlabel("z  [λ]"); ax[1].set_ylabel("|ψ| on the axis")
ax[1].legend(fontsize=9)
ttl(ax[1], "Interior field on the axis",
    f"a = {float(MC['a'])}λ, n₂ = {float(MC['n2'])}, k₂a = {float(MC['k2a']):.1f}, "
    f"N = {int(MC['N'])} · max rel. difference "
    f"{np.abs(MC['bem'] - MC['exact']).max() / np.abs(MC['exact']).max():.1e}")

for val, col, mk in ((1.5, BLUE, "o"), (2.5, VIOLET, "s")):
    m = MC["env_n2"] == val
    ax[2].semilogy(MC["env_k2a"][m], MC["env_err"][m], mk + "-", color=col,
                   lw=2.2, ms=7, label=f"n₂ = {val}")
ax[2].axhline(1e-2, color=MUTED, lw=1.4, ls=(0, (6, 4)))
ax[2].text(11, 1.3e-2, "1 %", fontsize=8.5, color=MUTED)
ax[2].set_xlabel("k₂a"); ax[2].set_ylabel("max rel. error in u")
ax[2].legend(fontsize=9, loc="lower right")
ttl(ax[2], "Error versus electrical size",
    "64–100 panels per interior wavelength throughout")
plt.tight_layout(); plt.show()

The error is set by $k_2a$, not by the contrast or by the mesh: at
64–100 panels per interior wavelength the surface error crosses 1 % around
$k_2a\approx 40$ for both indices, and the conditioning of the Müller
operator degrades past it.  Every ovoid run in these notebooks stays well
inside that range.